# Duplicate Removal: Part C (Tomato and Raspberry)

## Setup

In [8]:
import csv
import hashlib
import shutil
from collections import Counter
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from tqdm import tqdm

SOURCE_ROOT = Path("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color")
WORK_ROOT = Path("/kaggle/working/color")
SELECTED_CROPS = ["Tomato", "Raspberry"]
PART_NAME = "part_c"
HAMMING_THRESHOLD = 4
CONFLICT_RULE = "keep_one"
DRY_RUN = False
WRITE_DELETE_LOG = True
START_FRESH = True
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

BIT_COUNT_TABLE = np.array([bin(value).count("1") for value in range(256)], dtype=np.uint8)


## Functions

In [10]:
def crop_name(folder_name: str) -> str:
    return folder_name.split("___")[0]


def copy_selected_crops() -> None:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    for folder in sorted(SOURCE_ROOT.iterdir()):
        if not folder.is_dir() or crop_name(folder.name) not in SELECTED_CROPS:
            continue
        destination = WORK_ROOT / folder.name
        if destination.exists():
            print(f"Already copied: {folder.name}")
            continue
        print(f"Copying {folder.name}")
        shutil.copytree(folder, destination)


def list_image_files() -> list:
    files = [
        path for path in WORK_ROOT.rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS and crop_name(path.parent.name) in SELECTED_CROPS
    ]
    return sorted(files)


def compute_md5(path: Path) -> str:
    digest = hashlib.md5()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def build_records(files: list) -> list:
    records = []
    for path in tqdm(files, desc="Hashing"):
        try:
            with Image.open(path) as image:
                width, height = image.size
                perceptual_hash = int(str(imagehash.phash(image.convert("RGB"))), 16)
            records.append({
                "path": str(path.relative_to(WORK_ROOT.parent)),
                "crop": crop_name(path.parent.name),
                "class": path.parent.name,
                "md5": compute_md5(path),
                "phash": perceptual_hash,
                "width": width,
                "height": height,
            })
        except Exception as error:
            print(f"Skipped unreadable file {path}: {error}")
    return records


def count_differing_bits(values: np.ndarray) -> np.ndarray:
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(values)
    return BIT_COUNT_TABLE[values.view(np.uint8).reshape(-1, 8)].sum(axis=1)


def group_duplicates(records: list) -> list:
    count = len(records)
    hashes = np.array([record["phash"] for record in records], dtype=np.uint64)
    processing_order = sorted(
        range(count),
        key=lambda i: (-(records[i]["width"] * records[i]["height"]), records[i]["path"]),
    )

    keeper_hashes = np.zeros(count, dtype=np.uint64)
    groups = []
    for index in tqdm(processing_order, desc="Comparing"):
        keeper_count = len(groups)
        if keeper_count:
            distances = count_differing_bits(keeper_hashes[:keeper_count] ^ hashes[index])
            nearest = int(np.argmin(distances))
            if distances[nearest] <= HAMMING_THRESHOLD:
                groups[nearest].append(index)
                continue
        keeper_hashes[keeper_count] = hashes[index]
        groups.append([index])
    return groups


def select_deletions(records: list, groups: list) -> dict:
    deletions = {}
    for group in groups:
        if len(group) == 1:
            continue
        classes = {records[i]["class"] for i in group}
        has_conflict = len(classes) > 1
        if has_conflict and CONFLICT_RULE == "drop_all":
            for i in group:
                deletions[i] = "label conflict (all copies removed)"
        else:
            reason = "label conflict (best copy kept)" if has_conflict else "duplicate"
            for i in group[1:]:
                deletions[i] = reason
    return deletions


def write_csv(path: str, header: list, rows: list) -> None:
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)


def main() -> None:
    if CONFLICT_RULE not in ("drop_all", "keep_one"):
        raise ValueError("CONFLICT_RULE must be 'drop_all' or 'keep_one'")

    if START_FRESH and WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)

    copy_selected_crops()
    files = list_image_files()
    print(f"Images found for {SELECTED_CROPS}: {len(files)}")

    records = build_records(files)
    if not records:
        raise SystemExit("No readable images found. Check SOURCE_ROOT and SELECTED_CROPS.")

    groups = group_duplicates(records)
    deletions = select_deletions(records, groups)

    for index in tqdm(sorted(deletions), desc="Dry run" if DRY_RUN else "Deleting"):
        if not DRY_RUN:
            (WORK_ROOT.parent / records[index]["path"]).unlink(missing_ok=True)

    remaining = [record for i, record in enumerate(records) if i not in deletions]
    columns = list(records[0].keys())
    hash_file = Path(f"{PART_NAME}_hashes.csv")
    write_csv(str(hash_file), columns, [[r[c] for c in columns] for r in remaining])

    if WRITE_DELETE_LOG:
        write_csv(
            f"{PART_NAME}_deleted_log.csv",
            ["path", "reason"],
            [[records[i]["path"], deletions[i]] for i in sorted(deletions)],
        )

    print("\nSummary")
    print(f"Before: {len(records)} | Deleted: {len(deletions)} | After: {len(remaining)}")
    for reason, total in Counter(deletions.values()).items():
        print(f"  {reason}: {total}")
    if DRY_RUN:
        print("Dry run: no files were deleted.")
    print(f"Saved: {hash_file.resolve()}")


## Run

In [11]:
main()


Copying Raspberry___healthy
Copying Tomato___Bacterial_spot
Copying Tomato___Early_blight
Copying Tomato___Late_blight
Copying Tomato___Leaf_Mold
Copying Tomato___Septoria_leaf_spot
Copying Tomato___Spider_mites Two-spotted_spider_mite
Copying Tomato___Target_Spot
Copying Tomato___Tomato_Yellow_Leaf_Curl_Virus
Copying Tomato___Tomato_mosaic_virus
Copying Tomato___healthy
Images found for ['Tomato', 'Raspberry']: 18531


Deleting: 100%|██████████| 83/83 [00:00<00:00, 22207.66it/s]



Summary
Before: 18531 | Deleted: 83 | After: 18448
  duplicate: 76
  label conflict (best copy kept): 7
Saved: /kaggle/working/part_c_hashes.csv


## Deleted Images

In [12]:
deleted_log = pd.read_csv(f"{PART_NAME}_deleted_log.csv")
display(deleted_log["reason"].value_counts().rename("images").to_frame())
deleted_log.head(10)


,images
reason,
duplicate,76
label conflict (best copy kept),7


,path,reason
0,color/Raspberry___healthy/4c210242-adfc-4011-8...,duplicate
1,color/Raspberry___healthy/7ba1a91f-41d0-4ff2-a...,duplicate
2,color/Raspberry___healthy/7f771790-da66-43db-a...,duplicate
3,color/Raspberry___healthy/7fdeb60a-15db-4759-8...,duplicate
4,color/Raspberry___healthy/9f63daed-c468-4783-9...,duplicate
5,color/Raspberry___healthy/a8530446-f16a-4c42-a...,duplicate
6,color/Raspberry___healthy/ae5a17be-bbf6-47dd-a...,duplicate
7,color/Raspberry___healthy/aebc17a4-00bc-4a37-9...,duplicate
8,color/Raspberry___healthy/b00a6846-c092-4461-8...,duplicate
9,color/Raspberry___healthy/b4e5bed9-5737-4903-b...,duplicate


## Unique Images

In [13]:
unique_images = pd.read_csv(f"{PART_NAME}_hashes.csv")
files_on_disk = {str(path.relative_to(WORK_ROOT.parent)) for path in list_image_files()}

print(f"Unique images listed: {len(unique_images)}")
if DRY_RUN:
    print("Dry run: file check skipped")
else:
    print(f"Matches the files left on disk: {set(unique_images['path']) == files_on_disk}")
display(unique_images["class"].value_counts().rename("images").to_frame())
unique_images.head(10)


Unique images listed: 18448
Matches the files left on disk: True


,images
class,
Tomato___Tomato_Yellow_Leaf_Curl_Virus,5329
Tomato___Bacterial_spot,2120
Tomato___Late_blight,1900
Tomato___Septoria_leaf_spot,1771
Tomato___Spider_mites Two-spotted_spider_mite,1669
Tomato___healthy,1578
Tomato___Target_Spot,1403
Tomato___Early_blight,1000
Tomato___Leaf_Mold,951


,path,crop,class,md5,phash,width,height
0,color/Raspberry___healthy/00a3fc0e-64cc-4e35-a...,Raspberry,Raspberry___healthy,d1c650aa52fc1d3d09be5e203896a99d,18044464262181787277,256,256
1,color/Raspberry___healthy/020e14fb-6016-4d7f-b...,Raspberry,Raspberry___healthy,cdb580a0f29b9c766ce0948d09b97192,12654997694975623777,256,256
2,color/Raspberry___healthy/03a615df-ed85-4ae6-8...,Raspberry,Raspberry___healthy,5ce0d7a959abc451c6cdd237fab0ad6a,13452797524502616873,256,256
3,color/Raspberry___healthy/04908211-66c9-42f8-a...,Raspberry,Raspberry___healthy,5e23704143ddbc785c0c172749dd3f42,13433043158007796246,256,256
4,color/Raspberry___healthy/05a9ae7e-468f-4cb6-a...,Raspberry,Raspberry___healthy,1c7466d9254f72bad3a18716817ea125,16918199842079168101,256,256
5,color/Raspberry___healthy/05dfc382-396a-4301-b...,Raspberry,Raspberry___healthy,955c01d51a54e70ccc6e119cdfc0da5d,13805715709577180784,256,256
6,color/Raspberry___healthy/05f26de9-3f26-44f8-9...,Raspberry,Raspberry___healthy,90a7266274c17f90dfefb98a1612bc82,16915666572121269031,256,256
7,color/Raspberry___healthy/06959ec3-a76b-4968-b...,Raspberry,Raspberry___healthy,c7295c964cd59b5ad33b8ac48806cfaf,17275954472517669499,256,256
8,color/Raspberry___healthy/076d909d-cfca-47ce-a...,Raspberry,Raspberry___healthy,e7b7fe8e9b50cb7ad6249e5debe8c0f3,12360128484762224161,256,256
9,color/Raspberry___healthy/07871e60-96ce-45fb-b...,Raspberry,Raspberry___healthy,ef72987e9ac9a802cd2a588e6eaa6451,18357095529545761085,256,256
